# TerraFM LULC Segmentation — S1 + S2  (Google Colab)

**Every new session: run Cells 1 → 2 → 3 → 4, then either Cell 5 (first time) or Cell 6 (resume).**

Dataset structure after extraction:
```
/content/content/dataset/
    images/  S1/  S2/  reference_maps_selected/
    file_clean.csv
```

In [ ]:
# ── Cell 1 : Install dependencies ────────────────────────────────────
# Re-run at the start of EVERY session.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'rasterio>=1.3.9', 'timm>=0.9.12', 'transformers>=4.40.0',
    'huggingface_hub>=0.22.0', 'gdown', 'tqdm',
    'pandas', 'matplotlib', 'scikit-learn'
], check=True)
print('✓ Dependencies installed.')

In [ ]:
# ── Cell 2 : Clone project repo from GitHub ───────────────────────────
# Re-run at the start of EVERY session.
import os, subprocess

REPO_URL    = 'https://github.com/jenil1236/terrafm_lulc_segmentation.git'
PROJECT_DIR = '/content/terrafm_lulc_segmentation'

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
    print(f'✓ Cloned to {PROJECT_DIR}')
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)
    print('✓ Repo already present — pulled latest.')

In [ ]:
# ── Cell 3 : Download dataset from Google Drive ───────────────────────
# Run ONCE per session. Skips automatically if already extracted.
#
# How to get your file ID:
#   Drive → right-click zip → Share → Anyone with link
#   URL: https://drive.google.com/file/d/FILE_ID/view
#   Copy the FILE_ID part.

GDRIVE_FILE_ID = 'YOUR_GDRIVE_FILE_ID_HERE'   # ← paste your file ID here

import os, gdown, subprocess

MARKER = '/content/content/dataset/file_clean.csv'

if not os.path.exists(MARKER):
    ZIP = '/content/dataset.zip'
    gdown.download(id=GDRIVE_FILE_ID, output=ZIP, quiet=False)
    subprocess.run(['unzip', '-q', ZIP, '-d', '/content/'], check=True)
    print('✓ Dataset extracted.')
else:
    print('✓ Dataset already present — skipping download.')

import pandas as pd
df = pd.read_csv(MARKER)
print(f'  Rows in file_clean.csv: {len(df)}')

In [ ]:
# ── Cell 4 : Configure paths and import CFG ───────────────────────────
# Re-run at the start of EVERY session.
import sys, os
sys.path.insert(0, '/content/terrafm_lulc_segmentation')

from config import CFG
from utils import setup_logging, check_amp_support, save_config
from visualize import BIGEARTHNET_CLASS_NAMES, BIGEARTHNET_COLORS

setup_logging('INFO')

# Data paths
CFG.data_root = '/content/content/dataset/images'
CFG.s2_dir    = '/content/content/dataset/images/S2'
CFG.s1_dir    = '/content/content/dataset/images/S1'
CFG.ref_dir   = '/content/content/dataset/images/reference_maps_selected'
CFG.csv_file  = '/content/content/dataset/file_clean.csv'

# Output paths
CFG.output_dir     = '/content/outputs'
CFG.checkpoint_dir = '/content/outputs/checkpoints'
CFG.results_dir    = '/content/outputs/results'
CFG.split_dir      = '/content/outputs/splits'
CFG.weights_dir    = '/content/outputs/weights'
CFG.norm_stats_file    = '/content/outputs/norm_stats.json'
CFG.class_mapping_file = '/content/outputs/class_mapping.json'
CFG.final_model_path   = '/content/outputs/terrafm_lulc_model.pth'

# BigEarthNet class names and fixed colors
CFG.class_names  = BIGEARTHNET_CLASS_NAMES
CFG.class_colors = BIGEARTHNET_COLORS

# AMP — FP16 for T4
amp = check_amp_support()
CFG.amp_dtype = amp if amp != 'none' else 'fp16'

CFG.ensure_dirs()
save_config(CFG, '/content/outputs/config.json')
print(f'✓ Config ready.  AMP={CFG.amp_dtype}  |  Input channels={CFG.total_in_channels}')
print(f'  S2 : {CFG.s2_dir}')
print(f'  S1 : {CFG.s1_dir}')
print(f'  Ref: {CFG.ref_dir}')

In [ ]:
# ── Cell 5 : Prepare dataset ──────────────────────────────────────────
# Run ONCE (first time only, takes 5-10 min).
# On all subsequent sessions skip this and run Cell 6 instead.
from prepare_dataset import prepare_all

prep = prepare_all(csv_file=CFG.csv_file)

TRAIN_CSV    = prep['train_csv']
VAL_CSV      = prep['val_csv']
TEST_CSV     = prep['test_csv']
S2_MEAN      = prep['s2_mean']
S2_STD       = prep['s2_std']
S1_MEAN      = prep['s1_mean']
S1_STD       = prep['s1_std']
RAW_TO_TRAIN = prep['raw_to_train']
PIXEL_COUNTS = prep['pixel_counts']

print(f'\n✓ Preparation complete.')
print(f'  Train={TRAIN_CSV}')
print(f'  Val  ={VAL_CSV}')
print(f'  Test ={TEST_CSV}')

In [ ]:
# ── Cell 6 : Reload stats after session restart ───────────────────────
# Use this on every session AFTER Cell 5 has been run once.
# Before running this, upload your saved outputs/ folder to /content/outputs/
import json, os
from utils import load_class_mapping, load_class_stats

TRAIN_CSV = os.path.join(CFG.split_dir, 'train.csv')
VAL_CSV   = os.path.join(CFG.split_dir, 'val.csv')
TEST_CSV  = os.path.join(CFG.split_dir, 'test.csv')

with open(CFG.norm_stats_file) as f:
    ns = json.load(f)
S2_MEAN = ns['s2_mean'];  S2_STD = ns['s2_std']
S1_MEAN = ns['s1_mean'];  S1_STD = ns['s1_std']

RAW_TO_TRAIN = load_class_mapping(CFG.class_mapping_file)
PIXEL_COUNTS = load_class_stats(
    os.path.join(CFG.output_dir, 'class_stats.json')
)['pixel_counts']

print('✓ Stats reloaded.')
print(f'  S2 mean B01-B03 : {[round(v,4) for v in S2_MEAN[:3]]}')
print(f'  S1 mean (dB)    : {[round(v,4) for v in S1_MEAN]}')

In [ ]:
# ── Cell 7 : Visualize a reference map ───────────────────────────────
import pandas as pd
from visualize import plot_reference_map
from IPython.display import Image

row = pd.read_csv(CFG.csv_file).iloc[0]
ref_path = f"{CFG.ref_dir}/{row['reference_map_id']}.tif"
plot_reference_map(ref_path, save_path='/content/outputs/ref_preview.png')
Image('/content/outputs/ref_preview.png')

In [ ]:
# ── Cell 8 : Visualize training examples (S2 RGB + GT mask) ──────────
from dataset import S1S2LULCDataset, load_records_from_csv
from visualize import plot_qualitative_sample
from IPython.display import Image

records = load_records_from_csv(TRAIN_CSV)[:2]
ds = S1S2LULCDataset(
    records=records,
    s2_root=CFG.s2_dir, s1_root=CFG.s1_dir, ref_root=CFG.ref_dir,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN, augment=False,
)
fused, mask = ds[0]
plot_qualitative_sample(
    s2=fused[:12], gt_mask=mask, pred_mask=mask,
    class_names=CFG.class_names, class_colors=CFG.class_colors,
    ignore_index=CFG.ignore_index,
    save_path='/content/outputs/preview_0.png',
    norm_mean=S2_MEAN, norm_std=S2_STD,
)
Image('/content/outputs/preview_0.png')

In [ ]:
# ── Cell 9 : Phase 0 — Smoke test (MANDATORY) ────────────────────────
# Overfits 64 samples end-to-end.
# SUCCESS = train loss is a real number AND drops below 1.0 within 30 epochs.
# NaN loss = something is broken. Fix before proceeding.
from train import run_smoke_test

run_smoke_test(
    train_csv=TRAIN_CSV, val_csv=VAL_CSV,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN,
)

In [ ]:
# ── Cell 10 : Phase 1 — Pilot training (~2000 samples, 10 epochs) ─────
import pandas as pd, tempfile, os
from train import train as run_training

df = pd.read_csv(TRAIN_CSV).head(CFG.phase1_samples)
with tempfile.TemporaryDirectory() as tmp:
    p_csv = os.path.join(tmp, 'pilot.csv')
    df.to_csv(p_csv, index=False)
    run_training(
        train_csv=p_csv, val_csv=VAL_CSV,
        s2_mean=S2_MEAN, s2_std=S2_STD,
        s1_mean=S1_MEAN, s1_std=S1_STD,
        raw_to_train=RAW_TO_TRAIN, class_pixel_counts=PIXEL_COUNTS,
        epochs=CFG.phase1_epochs, batch_size=CFG.phase1_batch_size,
        freeze_stage=0, phase_name='phase1',
    )

In [ ]:
# ── Cell 11 : Phase 2 — Full training ────────────────────────────────
# Epochs 1-5  : encoder frozen (decoder warms up)
# Epoch  6+   : last 4 ViT blocks + patch embed unfrozen
# Early stop  : patience = 8 on val mIoU
#
# To RESUME after a crash:
#   Set RESUME_FROM = '/content/outputs/checkpoints/last_checkpoint.pth'
from train import train as run_training

RESUME_FROM = None   # ← set to checkpoint path to resume

best_ckpt = run_training(
    train_csv=TRAIN_CSV, val_csv=VAL_CSV,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN, class_pixel_counts=PIXEL_COUNTS,
    epochs=CFG.epochs, batch_size=CFG.batch_size,
    freeze_stage=0, phase_name='phase2',
    resume_from=RESUME_FROM,
)
print(f'Best checkpoint: {best_ckpt}')

In [ ]:
# ── Cell 12 : Evaluate on test set ───────────────────────────────────
from evaluate import evaluate
import os

results = evaluate(
    model_path=os.path.join(CFG.checkpoint_dir, 'best_model.pth'),
    test_csv=TEST_CSV,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN,
    class_names=CFG.class_names,
)
print(f'Test mIoU : {results["mean_iou"]:.4f}')
print(f'Test Dice : {results["mean_dice"]:.4f}')

In [ ]:
# ── Cell 13 : Plot training curves ───────────────────────────────────
from visualize import plot_training_curves
from IPython.display import Image

plot_training_curves(
    '/content/outputs/training_history_phase2.csv',
    '/content/outputs/results/training_curves.png',
)
Image('/content/outputs/results/training_curves.png')

In [ ]:
# ── Cell 14 : Export final model (terrafm_lulc_model.pth) ────────────
# Downloads the file to your machine automatically.
import os
from model import build_model, save_final_model
from checkpoint import load_checkpoint

model = build_model(freeze_stage=2)
load_checkpoint(os.path.join(CFG.checkpoint_dir, 'best_model.pth'), model)

save_final_model(
    model=model,
    path=CFG.final_model_path,
    s2_mean=S2_MEAN, s2_std=S2_STD,
    s1_mean=S1_MEAN, s1_std=S1_STD,
    raw_to_train=RAW_TO_TRAIN,
    class_names=CFG.class_names,
    class_colors=CFG.class_colors,
)
print(f'✓ Final model: {CFG.final_model_path}')

from google.colab import files
files.download(CFG.final_model_path)

In [ ]:
# ── Cell 15 : Inference on a new S1+S2 patch ─────────────────────────
from inference import patch_inference
from IPython.display import Image

# ← Set to your patch folder names inside S2/ and S1/
NEW_S2_DIR = f"{CFG.s2_dir}/S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_37_88"
NEW_S1_DIR = f"{CFG.s1_dir}/S1B_IW_GRDH_1SDV_20170612T165809_33UUP_37_88"

outputs = patch_inference(
    model_path=CFG.final_model_path,
    s2_patch_dir=NEW_S2_DIR,
    s1_patch_dir=NEW_S1_DIR,
    output_dir='/content/outputs/inference',
)
Image(outputs['vis_png'])

---
## Session Resume Cheatsheet

| Phase | Download before session ends | Why |
|---|---|---|
| After Cell 5 | `outputs/splits/train.csv`, `val.csv`, `test.csv` | Reproducible split |
| After Cell 5 | `outputs/norm_stats.json` | Must match training exactly |
| After Cell 5 | `outputs/class_mapping.json` + `class_stats.json` | Class IDs + loss weights |
| During Cell 11 | `outputs/checkpoints/last_checkpoint.pth` | Resume from exact epoch |
| After Cell 11 | `outputs/checkpoints/best_model.pth` | Best weights |
| After Cell 14 | `outputs/terrafm_lulc_model.pth` | Final inference model |

**New session steps:**
1. Run Cells 1 → 2 → 3 → 4 (always required)
2. Upload your saved `outputs/` folder to `/content/outputs/`
3. Run Cell 6 (skip Cell 5)
4. For Phase 2 resume: set `RESUME_FROM` in Cell 11 and run it